# 02_prompts

对应 `index.md` 第 2 阶段：PromptTemplate / ChatPromptTemplate。

笔记本在 `codes/langchain/02_prompts/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "02_prompts" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "02_prompts"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/02_prompts')

## 1. PromptTemplate 填空

用 `PromptTemplate.from_template` 建模板：`城市 {city} 的别称是 {nick}。`

用 `format` 填入 `city="重庆"`、`nick="山城"`，打印得到的字符串。


In [5]:
from langchain_core.prompts import PromptTemplate

# 作答

template = PromptTemplate.from_template("城市{city}的别称是{nick}。")

print(template.format(city="重庆", nick="山城"))
print(template.invoke({"city": "重庆", "nick": "山城"}))
#评阅
# format 用法对。题干模板是「城市 {city} 的别称是 {nick}。」，变量两侧有空格；你写成了无空格版。

#参考答案
# prompt = PromptTemplate.from_template("城市 {city} 的别称是 {nick}。")
# print(prompt.format(city="重庆", nick="山城"))

城市重庆的别称是山城。
text='城市重庆的别称是山城。'


## 2. ChatPromptTemplate 两条消息

用 `ChatPromptTemplate.from_messages` 组装：

- system：`你是{role}。`
- human：`请用一句话解释 {topic}。`

用 `invoke({"role": "老师", "topic": "引力"})`，打印结果里每条消息的类型名和 `content`。


In [9]:
from langchain_core.prompts import ChatPromptTemplate

# 作答

template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是{role}。"),
        ("human", "请用一句话解释 {topic}。"),
    ]
)

print(template.format(role="老师", topic="引力"))
print(template.invoke({"role": "老师", "topic": "引力"}))
#评阅
# 模板组装对。题要求打印每条消息的类型名和 content；应先 invoke，再对 messages 循环打印，而不是只 dump 整个对象。

#参考答案
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "你是{role}。"),
#     ("human", "请用一句话解释 {topic}。"),
# ])
# result = prompt.invoke({"role": "老师", "topic": "引力"})
# for m in result.to_messages():
#     print(type(m).__name__, m.content)

System: 你是老师。
Human: 请用一句话解释 引力。
messages=[SystemMessage(content='你是老师。', additional_kwargs={}, response_metadata={}), HumanMessage(content='请用一句话解释 引力。', additional_kwargs={}, response_metadata={})]


## 3. partial 预填一部分

已有模板 `prompt`，变量是 `product` 与 `tone`。

先 `partial(tone="正式")` 得到新模板，再只传入 `product="水壶"` 做 `format`，打印结果字符串。


In [10]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("用{tone}语气介绍产品：{product}")

# 作答

new_prompt = prompt.partial(tone="正式")
print(new_prompt.format(product="水壶"))
print(new_prompt.invoke({"product": "水壶"}))
#评阅
# 对。partial 预填 tone 后再 format product。

#参考答案
# print(prompt.partial(tone="正式").format(product="水壶"))

用正式语气介绍产品：水壶
text='用正式语气介绍产品：水壶'


## 4. MessagesPlaceholder 插入历史

用 `ChatPromptTemplate.from_messages` 组装：

- system：`你是助手。`
- `MessagesPlaceholder("history")`
- human：`{question}`

`invoke` 时传入已有的 `history` 和 `question="还要点什么？"`。打印最终消息条数，以及最后一条的 `content`。


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder

history = [
    HumanMessage(content="我想点咖啡"),
    AIMessage(content="好的，要什么口味？"),
]

# 作答

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="你是助手。"),
    MessagesPlaceholder(variable_name="history"),
    HumanMessagePromptTemplate.from_template("{question}") # HumanMessage(content="{question}") 会被错误当成字面消息
])

print("-" * 20)
print(prompt.format(history=history, question="还要点什么？"))
print("-" * 20)
print(prompt.format_messages(history=history, question="还要点什么？"))
print("-" * 20)
print(prompt.invoke({"history": history, "question": "还要点什么？"}))
print("-" * 20)
#评阅
# 组装对（Placeholder + HumanMessagePromptTemplate）。缺题要求的两行：消息条数、最后一条 content。

#参考答案
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "你是助手。"),
#     MessagesPlaceholder("history"),
#     ("human", "{question}"),
# ])
# msgs = prompt.invoke({"history": history, "question": "还要点什么？"}).to_messages()
# print(len(msgs))
# print(msgs[-1].content)

--------------------
System: 你是助手。
Human: 我想点咖啡
AI: 好的，要什么口味？
Human: 还要点什么？
--------------------
[SystemMessage(content='你是助手。', additional_kwargs={}, response_metadata={}), HumanMessage(content='我想点咖啡', additional_kwargs={}, response_metadata={}), AIMessage(content='好的，要什么口味？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='还要点什么？', additional_kwargs={}, response_metadata={})]
--------------------
messages=[SystemMessage(content='你是助手。', additional_kwargs={}, response_metadata={}), HumanMessage(content='我想点咖啡', additional_kwargs={}, response_metadata={}), AIMessage(content='好的，要什么口味？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='还要点什么？', additional_kwargs={}, response_metadata={})]
--------------------


## 5. 模板接到假模型

已有 `prompt`（ChatPromptTemplate）和 `model`（假模型）。

先 `prompt.invoke({"name": "小明"})` 得到消息，再交给 `model.invoke(...)`，打印模型返回的 `content`。


In [26]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你只回应用户的名字。"),
        ("human", "我叫{name}"),
    ]
)
model = FakeListChatModel(responses=["小明"])

# 作答

messages = prompt.invoke({"name": "小明"})
print(type(messages).__name__)
print(type(messages.messages).__name__)
print(type(messages.messages[0]).__name__)
print(model.invoke(messages).content)
#评阅
# 对。prompt.invoke 后再 model.invoke，content 为「小明」。

#参考答案
# msgs = prompt.invoke({"name": "小明"})
# print(model.invoke(msgs).content)

ChatPromptValue
list
SystemMessage
小明
